# Tutorial 1 - Finding representations for new samples

This tutorial shows how to find the best representations in latent space for a new set of samples, using a pre-trained bulkdgd model. We start from a CSV file of raw gene expression samples and end up with their latent representations and the decoder's predicted gene expression for each of them.

In [ ]:
# Import from the standard library.
import logging as log

# Import from third-party libraries.
import pandas as pd

# Import from 'bulkdgd'.
from bulkdgd import ioutil
from bulkdgd.core import model

# Set the logging options so that every message of level INFO or above
# is emitted.
log.basicConfig(level = "INFO")

## Preprocess the samples

Load the raw samples from a CSV file, then preprocess them so that their genes match the set of genes included in the bulkdgd model.

In [2]:
# Load the samples into a data frame.
df_samples = \
    ioutil.load_samples(# The CSV file where the samples are stored
                        csv_file = "samples.csv",
                        # The field separator used in the CSV file
                        sep = ",",
                        # Whether to keep the original samples' names/
                        # indexes (if True, they are assumed to be in
                        # the first column of the data frame)
                        keep_samples_names = True,
                        # Whether to split the input data frame into
                        # two data frames, one containing only gene
                        # expression data and the other containing
                        # additional information about the samples                  
                        split = False)

# Pre-process the samples.
df_preproc, genes_excluded, genes_missing = \
    ioutil.preprocess_samples(df_samples = df_samples)

INFO:bulkdgd.ioutil.samplesio:Since 'keep_samples_names = True', the samples will be identified using the values contained in the first column of the data frame.


INFO:bulkdgd.ioutil.samplesio:16883 column(s) containing gene expression data was (were) found in the input data frame.


INFO:bulkdgd.ioutil.samplesio:16883 column(s) containing gene expression data was (were) found in the input data frame.


INFO:bulkdgd.ioutil.samplesio:1 column(s) containing additional information (not gene expression data) was (were) in the input data frame: tissue.


INFO:bulkdgd.ioutil.samplesio:Now looking for duplicated samples...


...

INFO:bulkdgd.ioutil.samplesio:No missing values were found in the columns containing gene expression data.


INFO:bulkdgd.ioutil.samplesio:Now looking for duplicated genes...


INFO:bulkdgd.ioutil.samplesio:No duplicated genes were found.


INFO:bulkdgd.ioutil.samplesio:In the data frame containing the pre-processed samples, the columns containing gene expression data will be ordered according to the list of genes included in the DGD model (taken from '/home/lcb518/software/bulkdgd-2.0.0/bulkdgd/data/model/genes/genes.txt').


INFO:bulkdgd.ioutil.samplesio:In the data frame containing the pre-processed samples, the columns found in the input data frame which did not contain gene expression data, if any were present, will be appended as the last columns of the data frame and appear in the same order as they did in the input data frame.


## Load the configurations

Load the configuration describing the pre-trained model's architecture and the default configuration for the two-round optimization scheme used to find representations.

In [3]:
# Load the configuration for the trained model.
config_model = ioutil.load_config_model("model_tgmm_trained")

# Load the default configuration to find the best representations.
config_rep = ioutil.load_config_rep(None)

## Get the trained model

Instantiate the bulkdgd model (Gaussian mixture latent space and decoder), loading the pre-trained parameters referenced in the model configuration.

In [4]:
# Get the trained bulkDGD model (latent space and decoder).
dgd_model = model.BulkDGD(**config_model)

INFO:bulkdgd.core.model:The parameters were successfully loaded from '/home/lcb518/software/bulkdgd-2.0.0/bulkdgd/data/model/gmm/gmm.pth'.


INFO:bulkdgd.core.model:The latent space was successfully set (type: 'GaussianMixtureModelTGMM').


INFO:bulkdgd.core.decoders:The decoder's hidden layer # 1 was set. Input features: 32. Output features: 500. Activation function: 'ReLU'.


INFO:bulkdgd.core.decoders:The decoder's hidden layer # 2 was set. Input features: 500. Output features: 8000. Activation function: 'ReLU'.


INFO:bulkdgd.core.decoders:The decoder's output module was set. Module 'OutputModuleNBFeatureDispersion' (activation = 'softplus', r_init = 2, output_dim = 14895).


INFO:bulkdgd.core.decoders:The decoder's dropout fraction was set to 0.0.


INFO:bulkdgd.core.model:The parameters were successfully loaded from '/home/lcb518/software/bulkdgd-2.0.0/bulkdgd/data/model/dec/dec.pth'.


INFO:bulkdgd.core.model:The decoder was successfully set.


## Get the representations

For each preprocessed sample, find the representation in latent space that best reconstructs its gene expression profile, along with the decoder's predicted means (and r-values, if applicable) for that representation.

In [5]:
# Get the representations, the predicted scaled means and
# r-values of the negative binomials modeling the genes' counts for
# the in silico samples corresponding to the representations found,
# and the time spent finding the representations.
df_rep, df_pred_means, df_pred_r_values, df_time_opt = \
    dgd_model.get_representations(\
        # The data frame with the pre-processed samples
        df_samples = df_preproc,
        # The configuration to find the representations
        config_rep = config_rep)

INFO:bulkdgd.core.model:Starting optimization number 1...


INFO:bulkdgd.core.model:Epoch 1: loss 45603980.847, epoch CPU time 94.978 s, backward step CPU time 69.156 s, epoch wall clock time 1.993 s, backward step wall clock time 1.567 s.


INFO:bulkdgd.core.model:Epoch 2: loss 42951241.813, epoch CPU time 63.767 s, backward step CPU time 46.601 s, epoch wall clock time 1.269 s, backward step wall clock time 0.994 s.


INFO:bulkdgd.core.model:Epoch 3: loss 40717904.042, epoch CPU time 65.341 s, backward step CPU time 47.581 s, epoch wall clock time 1.289 s, backward step wall clock time 1.004 s.


INFO:bulkdgd.core.model:Epoch 4: loss 38804133.988, epoch CPU time 63.579 s, backward step CPU time 45.744 s, epoch wall clock time 1.233 s, backward step wall clock time 0.941 s.


...

INFO:bulkdgd.core.model:Epoch 46: loss 524156.123, epoch CPU time 37.864 s, backward step CPU time 32.168 s, epoch wall clock time 0.799 s, backward step wall clock time 0.708 s.


INFO:bulkdgd.core.model:Epoch 47: loss 524059.030, epoch CPU time 37.329 s, backward step CPU time 31.559 s, epoch wall clock time 0.782 s, backward step wall clock time 0.691 s.


INFO:bulkdgd.core.model:Epoch 48: loss 523960.465, epoch CPU time 37.103 s, backward step CPU time 31.402 s, epoch wall clock time 0.778 s, backward step wall clock time 0.689 s.


INFO:bulkdgd.core.model:Epoch 49: loss 523863.397, epoch CPU time 36.978 s, backward step CPU time 31.319 s, epoch wall clock time 0.763 s, backward step wall clock time 0.676 s.


INFO:bulkdgd.core.model:Epoch 50: loss 523773.521, epoch CPU time 36.867 s, backward step CPU time 31.238 s, epoch wall clock time 0.765 s, backward step wall clock time 0.676 s.


## Save the outputs

Save the preprocessed samples, the representations found, the decoder's predictions, and the time spent on the optimization to CSV files.

In [6]:
# Save the pre-processed samples.
ioutil.save_samples(\
   # The data frame containing the samples
   df = df_preproc,
   # The output CSV file
   csv_file = "samples_preprocessed.csv",
   # The field separator in the output CSV file
   sep = ",")

# Save the representations.
ioutil.save_representations(\
    # The data frame containing the representations
    df = df_rep,
    # The output CSV file
    csv_file = "representations.csv",
    # The field separator in the output CSV file
    sep = ",")

# Save the predicted scaled means.
ioutil.save_decoder_outputs(\
    # The data frame containing the predicted scaled means
    df = df_pred_means,
    # The output CSV file
    csv_file = "pred_means.csv",
    # The field separator in the output CSV file
    sep = ",")

# Save the predicted r-values.
ioutil.save_decoder_outputs(\
    # The data frame containing the predicted r-values
    df = df_pred_r_values,
    # The output CSV file
    csv_file = "pred_r_values.csv",
    # The field separator in the output CSV file
    sep = ",")

# Save the time data.
df_time_opt.to_csv("time_opt.csv",
                   index = True,
                   header = True,
                   sep = ",")